# 12 — RQ1 outcome multiverse

**Objective.** Estimate paired cross-specification distance, within-label refit noise, signed Δspec, NACF, hierarchical-bootstrap intervals, and three-way sensitivity/equivalence decisions.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

A non-significant contrast is never relabelled as stability; only a fully contained equivalence interval supports practical equivalence.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("12", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import numpy as np
import pandas as pd
import yaml
from cruxvc.inference import attribution_vectors, bootstrap_interval, hierarchical_bootstrap_rq1, rq1_delta_spec, three_way_interpretation
from cruxvc.io import read_table, write_table
from cruxvc.manifest import append_test_access_log

append_test_access_log(P, stage_id="12", purpose="locked RQ1 inference", resources=[P.attributions / "final_attributions_long.parquet"])
long = read_table(P.attributions / "final_attributions_long.parquet")
long = long[long["analysis_role"].eq("matched_reference_bootstrap")].copy()
wide = attribution_vectors(long)
statistical = yaml.safe_load((P.config / "statistical_analysis.yaml").read_text())
repetitions = int(statistical["inference"]["bootstrap_repetitions"])

In [ ]:
summaries = []
cross_parts = []
within_parts = []
bootstrap_parts = []
for index, (outcome_a, outcome_b) in enumerate([("F18", "F36"), ("F36", "B+36")]):
    summary, cross, within = rq1_delta_spec(
        wide, outcome_a, outcome_b,
        nacf_epsilon=float(CFG["explanations"]["nacf_denominator_epsilon"]),
    )
    draws = hierarchical_bootstrap_rq1(
        cross, within, repetitions=repetitions,
        seed=int(statistical["inference"]["bootstrap_seed"]) + index,
    )
    interval = bootstrap_interval(draws)
    decision = three_way_interpretation(
        summary["delta_spec"], interval,
        meaningful_effect=float(statistical["smallest_meaningful_effects"]["rq1_delta_spec"]),
        equivalence_half_width=float(statistical["power"]["equivalence_half_width"]),
    )
    summaries.append({
        "contrast": f"{outcome_a}_vs_{outcome_b}", **summary,
        "ci_lower": interval[0], "ci_upper": interval[1], "decision": decision,
        "bootstrap_repetitions": int(np.isfinite(draws).sum()),
        "one_sided_p_delta_le_zero": float((draws <= 0).mean()),
    })
    cross_parts.append(cross.assign(contrast=f"{outcome_a}_vs_{outcome_b}"))
    within_parts.append(within.assign(contrast=f"{outcome_a}_vs_{outcome_b}"))
    bootstrap_parts.append(pd.DataFrame({"contrast": f"{outcome_a}_vs_{outcome_b}", "draw": draws}))

In [ ]:
summary_frame = pd.DataFrame(summaries)
summary_path = write_table(summary_frame, P.inference / "rq1_specification_effects.csv")
cross_path = write_table(pd.concat(cross_parts, ignore_index=True), P.inference / "rq1_cross_spec_distances.parquet")
within_path = write_table(pd.concat(within_parts, ignore_index=True), P.inference / "rq1_within_refit_distances.parquet")
draws_path = write_table(pd.concat(bootstrap_parts, ignore_index=True), P.inference / "rq1_bootstrap_draws.parquet")
CTX.recorder.complete([summary_path, cross_path, within_path, draws_path])
print(summary_frame.to_string(index=False))